In [3]:
import pandas as pd
import numpy as np
 # load cleaned dataset
dataset = pd.read_csv('/content/placement_data_cleaned.csv')
dataset.head()

,Student_ID,Age,Gender,Degree,Branch,CGPA,Internships,Projects,Coding_Skills,Communication_Skills,Aptitude_Test_Score,Soft_Skills_Rating,Certifications,Backlogs,Placement_Status,Placement_Numeric
0,15202,23,Male,B.Tech,Civil,7.32,2,4,5,6,56,6,1,0,Placed,1
1,4573,24,Female,MCA,ME,4.76,0,1,1,4,37,4,0,3,Not Placed,0
2,34424,20,Male,BCA,ME,6.16,0,3,3,8,68,6,1,3,Not Placed,0
3,38881,19,Male,B.Sc,IT,8.77,2,5,8,5,83,6,3,0,Placed,1
4,30191,23,Male,B.Tech,ME,7.63,0,3,4,6,66,7,1,0,Not Placed,0


In [4]:
#encode gender and branch using map
dataset['Gender'] = dataset['Gender'].map({'Male': 1, 'Female': 0})
dataset = pd.get_dummies(dataset, columns=['Branch', 'Degree'], drop_first=True)
dataset.head()
dataset.columns


Index(['Student_ID', 'Age', 'Gender', 'CGPA', 'Internships', 'Projects',
       'Coding_Skills', 'Communication_Skills', 'Aptitude_Test_Score',
       'Soft_Skills_Rating', 'Certifications', 'Backlogs', 'Placement_Status',
       'Placement_Numeric', 'Branch_Civil', 'Branch_ECE', 'Branch_IT',
       'Branch_ME', 'Degree_B.Tech', 'Degree_BCA', 'Degree_MCA'],
      dtype='object')

In [5]:
# defining features(x) and target(y)
X = dataset.drop(['Student_ID', 'Placement_Status', 'Placement_Numeric'], axis=1)
y = dataset['Placement_Numeric']

In [6]:
# train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

(4000, 18) (1000, 18)


In [7]:
#Feature scaling( standardization= (value - mean) / std_dev)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train_scaled.mean(axis=0)[:5])
print(X_train_scaled.std(axis=0)[:5])

[-2.80664381e-16  1.49213975e-16  9.02389274e-16  1.42108547e-17
  1.84741111e-16]
[1. 1. 1. 1. 1.]


In [11]:
from sklearn.linear_model import LogisticRegression
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)

print(y_pred_log[:10])
print(y_test[:10].values)

[0 1 1 0 0 1 1 0 1 1]
[0 1 0 0 1 1 1 0 1 1]


In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
print("Accuracy:", accuracy_score(y_test, y_pred_log))
#True Positives / (True Positives + False Positives)
print("Precision:", precision_score(y_test, y_pred_log))

print("Recall:", recall_score(y_test, y_pred_log))
print("F1 Score:", f1_score(y_test, y_pred_log))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))

Accuracy: 0.871
Precision: 0.8442622950819673
Recall: 0.8110236220472441
F1 Score: 0.8273092369477911
Confusion Matrix:
 [[562  57]
 [ 72 309]]


In [17]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print(X.columns.tolist())


Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
 [[619   0]
 [  0 381]]
['Age', 'Gender', 'CGPA', 'Internships', 'Projects', 'Coding_Skills', 'Communication_Skills', 'Aptitude_Test_Score', 'Soft_Skills_Rating', 'Certifications', 'Backlogs', 'Branch_Civil', 'Branch_ECE', 'Branch_IT', 'Branch_ME', 'Degree_B.Tech', 'Degree_BCA', 'Degree_MCA']


In [25]:
#check authentication
print(X_train.duplicated().sum())
print(X_test.duplicated().sum())
combined = pd.concat([X_train, X_test])
print(combined.duplicated().sum())
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)
print(dataset.shape)
print(dataset['Placement_Numeric'].value_counts())
from sklearn.ensemble import RandomForestClassifier

rf_limited = RandomForestClassifier(max_depth=3, n_estimators=10, random_state=42)
rf_limited.fit(X_train_scaled, y_train)
y_pred_limited = rf_limited.predict(X_test_scaled)

print("Limited RF Accuracy:", accuracy_score(y_test, y_pred_limited))


0
0
0
(5000, 21)
Placement_Numeric
0    3188
1    1812
Name: count, dtype: int64
Limited RF Accuracy: 0.918


In [26]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, y_pred_log), accuracy_score(y_test, y_pred_rf)],
    'Precision': [precision_score(y_test, y_pred_log), precision_score(y_test, y_pred_rf)],
    'Recall': [recall_score(y_test, y_pred_log), recall_score(y_test, y_pred_rf)],
    'F1 Score': [f1_score(y_test, y_pred_log), f1_score(y_test, y_pred_rf)]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.871,0.844262,0.811024,0.827309
1,Random Forest,1.000,1.000000,1.000000,1.000000
